In [71]:
import requests
import pandas as pd
from datetime import datetime, timezone
import time
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.colors import BoundaryNorm
from matplotlib.cm import get_cmap
from matplotlib.patches import Patch
import matplotlib.dates as mdates
import ast
import folium
from folium.plugins import MarkerCluster
import reverse_geocoder as rg
import re
import pycountry
import os
import numpy as np
import geopandas as gpd
import fiona
import sys
from shapely.geometry import Point
from sklearn.cluster import DBSCAN
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RepeatedKFold, cross_val_score
from sklearn.inspection import permutation_importance
from sklearn.inspection import PartialDependenceDisplay
from sklearn.model_selection import LeaveOneOut
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
import ruptures as rpt
from haversine import haversine
import functions as own
from timezonefinder import TimezoneFinder
import zoneinfo
from itertools import product
from scipy.stats import spearmanr
import seaborn as sns
from geopy.distance import geodesic
import wbgapi as wb
from IPython.display import display
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

Distance to Zurich: Mean or Median distance of all observations in a country to Zurich (47.3717, 8.5422)

In [13]:
df = pd.read_csv("../CWData_clean7.csv")

Zurich_coords = (47.3717, 8.5422)

# mean and median coordinates per country
df["distance_to_zurich_km"] = df.apply(
    lambda row: geodesic((row["latitude"], row["longitude"]), Zurich_coords).km, axis=1
)

distance_per_country = (
    df.groupby("Country")
    .agg(
        total_observations=("created_by", "size"),
        distance_to_zurich_km_mean=("distance_to_zurich_km", "mean"),
        distance_to_zurich_km_median=("distance_to_zurich_km", "median"),
    )
    .reset_index()
)

distance_per_country = distance_per_country[distance_per_country["total_observations"] >= 10].reset_index(drop=True)

pd.set_option("display.max_rows", None)
distance_per_country

C:\Users\yanni\AppData\Local\Temp\ipykernel_38788\1866434761.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 

,Country,total_observations,distance_to_zurich_km_mean,distance_to_zurich_km_median
0,Argentina,110,12401.684666,13259.754221
1,Australia,357,16004.245814,16232.804061
2,Austria,6247,454.266481,360.414399
3,Belgium,18,395.622967,368.974988
4,Brazil,148,9366.525604,9423.963887
5,Bulgaria,25,1318.665636,1294.777419
6,Canada,2175,7538.529621,7854.139757
7,Chile,947,12341.412200,12338.268890
8,China,85,8036.545453,7437.518515
9,Colombia,15,8965.976002,8804.712298


Get population, population density, area, GDP per capita for each country

In [46]:
world_area = gpd.read_file("../Borders/ne_10m_admin_0_countries/ne_10m_admin_0_countries.shp", engine="fiona")
world_area = world_area.to_crs("ESRI:54009")
world_area["area_km2"] = world_area.geometry.area / 1e6
country_attrs = world_area[["ADM0_A3", "area_km2"]].copy()

indicators = {
    "SP.POP.TOTL": "population",
    "NY.GDP.PCAP.CD": "gdp_per_capita",
    "EN.POP.DNST": "pop_density",
}

wb_df = wb.data.DataFrame(list(indicators), time=2023, labels=False).reset_index()
wb_df = wb_df.rename(columns={"economy": "ADM0_A3", **indicators})

edu_df = pd.read_csv("../Education_Index/Education_index.csv", sep=";", encoding="utf-8-sig")
edu_df = edu_df[["ADM0_A3", "Education_Index"]]

regression_data = distance_per_country.copy()
regression_data["ADM0_A3"] = regression_data["Country"].apply(own.get_iso3)
regression_data = regression_data.merge(wb_df, on="ADM0_A3", how="left")
regression_data = regression_data.merge(country_attrs, on="ADM0_A3", how="left")
regression_data = regression_data.merge(edu_df, on="ADM0_A3", how="left")

regression_data = regression_data[regression_data["ADM0_A3"] != "CHE"]

regression_data

,Country,total_observations,distance_to_zurich_km_mean,distance_to_zurich_km_median,ADM0_A3,pop_density,gdp_per_capita,population,area_km2,Education_Index
0,Argentina,110,12401.684666,13259.754221,ARG,16.639956,14261.846567,4.553840e+07,2.789349e+06,0.872819
1,Australia,357,16004.245814,16232.804061,AUS,3.465919,65058.377315,2.665992e+07,7.723183e+06,0.929000
2,Austria,6247,454.266481,360.414399,AUT,110.661185,56579.504175,9.131761e+06,8.394346e+04,0.864390
3,Belgium,18,395.622967,368.974988,BEL,386.303732,55244.681458,1.177995e+07,3.062973e+04,0.922988
4,Brazil,148,9366.525604,9423.963887,BRA,25.261688,10377.589279,2.111407e+08,8.523997e+06,0.719532
5,Bulgaria,25,1318.665636,1294.777419,BGR,59.382793,15854.019289,6.446596e+06,1.128203e+05,0.807280
6,Canada,2175,7538.529621,7854.139757,CAN,4.556884,54847.537011,4.004909e+07,9.916644e+06,0.903674
7,Chile,947,12341.412200,12338.268890,CHL,26.464725,17081.518074,1.965884e+07,7.380176e+05,0.846158
8,China,85,8036.545453,7437.518515,CHN,150.264001,12951.178240,1.410710e+09,9.394406e+06,0.697838
9,Colombia,15,8965.976002,8804.712298,COL,47.157415,7012.491691,5.232115e+07,1.142648e+06,0.697897


Random forest regression with population, GDP per capita and median distance to Zurich as dependent, and number of observations as independent variable

Log transformation of number of observations

In [48]:
d = regression_data.dropna(subset=["total_observations", "population", "gdp_per_capita", "distance_to_zurich_km_median", "area_km2", "Education_Index"]).copy()

d["log_obs"] = np.log10(d["total_observations"])
d["log_pop"] = np.log10(d["population"])
d["log_gdp"] = np.log10(d["gdp_per_capita"])
d["log_dist"] = np.log10(d["distance_to_zurich_km_median"].clip(lower=1))
d["log_area"] = np.log10(d["area_km2"])

In [ ]:
features = ["log_pop", "log_gdp", "log_dist", "log_area", "Education_Index"]
X = d[features]
y = d["log_obs"]

# collinearity
display(X.corr().round(2))

,log_pop,log_gdp,log_dist,log_area,Education_Index
log_pop,1.00,-0.41,0.47,0.86,-0.31
log_gdp,-0.41,1.00,-0.55,-0.28,0.82
log_dist,0.47,-0.55,1.00,0.58,-0.47
log_area,0.86,-0.28,0.58,1.00,-0.12
Education_Index,-0.31,0.82,-0.47,-0.12,1.00


In [73]:
final_features = ["log_pop", "log_gdp", "log_dist"]
final_X = d[final_features]
final_y = d["log_obs"]

loo = LeaveOneOut()

rf = RandomForestRegressor(
    n_estimators=1000,
    min_samples_leaf=3,      #against overfitting
    max_features=2,
    random_state=1,
    oob_score=True,
)

y_true_rf, y_pred_rf = [], []
for train_idx, test_idx in loo.split(final_X):
    X_train, X_test = final_X.iloc[train_idx], final_X.iloc[test_idx]
    y_train, y_test = final_y.iloc[train_idx], final_y.iloc[test_idx]
    rf.fit(X_train, y_train)
    pred = rf.predict(X_test)
    y_true_rf.append(y_test.values[0])
    y_pred_rf.append(pred[0])

rf_loo_r2 = r2_score(y_true_rf, y_pred_rf)
print(f"RF LOO CV R²: {rf_loo_r2:.3f}")

rf.fit(final_X, final_y)
print(f"OOB R²: {rf.oob_score_:.3f}")

# Permutation Importance instead of impurity-based
perm = permutation_importance(rf, final_X, final_y, n_repeats=50, random_state=42, scoring="r2")
imp = pd.DataFrame({
    "feature": final_features,
    "importance": perm.importances_mean,
    "std": perm.importances_std,
}).sort_values("importance", ascending=False)
display(imp)

RF LOO CV R²: 0.143
OOB R²: 0.161


,feature,importance,std
1,log_gdp,0.752247,0.142724
0,log_pop,0.455328,0.075637
2,log_dist,0.136888,0.016758


In [80]:
n = len(final_features)
ncols = 2
nrows = -(-n // ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(10, 4*nrows))
axes = axes.flatten()

feature_labels = {
    "log_gdp": "log(GDP per capita)",
    "log_pop": "log(Population)",
    "log_dist": "log(Distance to Zurich)"
}

display = PartialDependenceDisplay.from_estimator(
    rf, final_X, final_features,
    ax=axes[:n],
    line_kw={"color": "teal", "linewidth": 2}
)

for ax, feat in zip(axes[:n], final_features):
    for coll in ax.collections:
        coll.remove()
    
    ax.set_xlabel(feature_labels.get(feat, feat), fontsize=13)
    ax.set_ylabel("Partial dependence", fontsize=13)
    ax.tick_params(labelsize=11)
    ax.grid(axis="both", linestyle="--", alpha=0.4)

for ax in axes[n:]:
    ax.axis("off")


fig.suptitle("Partial Dependence: Random Forest Regression (Country Level)", fontsize=16)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig("../Products/RF_partial_dependence.png", dpi=300, bbox_inches="tight")
plt.close()

"Normal" Multiple Regression

In [75]:
lr = LinearRegression()

y_true_lr, y_pred_lr = [], []
for train_idx, test_idx in loo.split(final_X):
    X_train, X_test = final_X.iloc[train_idx], final_X.iloc[test_idx]
    y_train, y_test = final_y.iloc[train_idx], final_y.iloc[test_idx]
    lr.fit(X_train, y_train)
    pred = lr.predict(X_test)
    y_true_lr.append(y_test.values[0])
    y_pred_lr.append(pred[0])

lr_loo_r2 = r2_score(y_true_lr, y_pred_lr)
print(f"OLS LOO CV R²: {lr_loo_r2:.3f}")

# statsmodels für Koeffizienten/p-Werte bleibt unverändert
X_ols = sm.add_constant(final_X)
ols_model = sm.OLS(final_y, X_ols).fit()
ols_model.summary()

OLS LOO CV R²: 0.076


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                log_obs   R-squared:                       0.231
Model:                            OLS   Adj. R-squared:                  0.185
Method:                 Least Squares   F-statistic:                     4.997
Date:                Fri, 25 Sep 2026   Prob (F-statistic):            0.00415
Time:                        08:20:14   Log-Likelihood:                -57.201
No. Observations:                  54   AIC:                             122.4
Df Residuals:                      50   BIC:                             130.4
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -3.6013      1.851     -1.946      0.057      -7.319       0.116
log_pop        0.3225      0.139      2.316      0.025       0.043       0.602
log_gdp        0.8291      0.252      3.296      0.002       0.324       1.334
log_dist      -0.0364      0.222     -0.164      0.870      -0.481       0.409
==============================================================================
Omnibus:                        2.622   Durbin-Watson:                   2.420
Prob(Omnibus):                  0.270   Jarque-Bera (JB):                1.970
Skew:                           0.300   Prob(JB):                        0.373
Kurtosis:                       2.282   Cond. No.                         173.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [76]:
X_vif = sm.add_constant(final_X)

vif_data = pd.DataFrame()
vif_data["feature"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
vif_data

,feature,VIF
0,const,351.614331
1,log_pop,1.346761
2,log_gdp,1.494941
3,log_dist,1.603446
